## 单元全局刚度矩阵

In [12]:
def stiffness_beam(EI, L):
    '''
    2自由度beam bending element二节点单元
    全局刚度矩阵
    '''
    k = EI / L**3 * np.array([[12, 6*L, -12, 6*L],
                              [6*L, 4*L**2, -6*L, 2*L**2],
                              [-12, -6*L, 12, -6*L],
                              [6*L, 2*L**2, -6*L, 4*L**2]])  
    
    return k

## 组装

In [13]:
def assemble_K(ke_list, connectivity, n_nodes, numbering="one"):
    """
    组装每节点 2 自由度单元的总体刚度矩阵

    适用于：
    1. 二维 pin-jointed truss: [u, v]
    2. 二维梁弯曲单元: [v, theta]

    Parameters
    ----------
    ke_list : list
        单元刚度矩阵列表，每个矩阵应为 4x4

    connectivity : list
        单元连接关系，例如 [(1, 2), (2, 3)]

    n_nodes : int
        总节点数

    numbering : str
        "one"  表示节点编号从 1 开始
        "zero" 表示节点编号从 0 开始

    Returns
    -------
    K : ndarray
        总体刚度矩阵
    """

    if len(ke_list) != len(connectivity):
        raise ValueError("ke_list and connectivity must have the same length.")

    total_dof = 2 * n_nodes
    K = np.zeros((total_dof, total_dof))

    for ke, conn in zip(ke_list, connectivity):

        ke = np.asarray(ke, dtype=float)

        if ke.shape != (4, 4):
            raise ValueError("Each element stiffness matrix must be 4x4.")

        ni, nj = conn

        if numbering == "one":
            ni -= 1
            nj -= 1
        elif numbering == "zero":
            pass
        else:
            raise ValueError("numbering must be 'one' or 'zero'.")

        dofs = [
            2 * ni,      # node i: v_i
            2 * ni + 1,  # node i: theta_i
            2 * nj,      # node j: v_j
            2 * nj + 1   # node j: theta_j
        ]

        K[np.ix_(dofs, dofs)] += ke

    return K

## 自由度编号定义

In [14]:
def dof_beam(node, direction, numbering="one"):
    """
    二维梁单元自由度编号

    每个节点两个自由度:
        v      : transverse displacement
        theta  : rotation
    """

    if numbering == "one":
        node = node - 1
    elif numbering == "zero":
        pass
    else:
        raise ValueError("numbering must be 'one' or 'zero'.")

    direction = direction.lower()

    if direction in ["v", "y"]:
        return 2 * node
    elif direction in ["theta", "th", "rotation", "rz"]:
        return 2 * node + 1
    else:
        raise ValueError("direction must be 'v' or 'theta'.")

## 求解

In [15]:
def solve_structure(K, force_bc=None, disp_bc=None):
    """
    求解总体刚度方程 K d = F

    Parameters
    ----------
    K : ndarray
        总体刚度矩阵

    force_bc : dict
        已知外力边界条件，格式为：
        {
            dof_index: force_value
        }

        例如：
        {
            3: -50
        }

    disp_bc : dict
        已知位移边界条件，格式为：
        {
            dof_index: displacement_value
        }

        例如：
        {
            0: 0,
            1: 0,
            4: 0,
            5: 0
        }

    Returns
    -------
    d : ndarray
        完整位移向量

    reaction : ndarray
        完整反力向量

    free_dofs : ndarray
        自由自由度编号

    fixed_dofs : ndarray
        约束自由度编号
    """

    K = np.asarray(K, dtype=float)
    n_dof = K.shape[0]

    if K.shape[0] != K.shape[1]:
        raise ValueError("K must be a square matrix.")

    # -------------------------
    # 1. 定义总外力向量 F
    # -------------------------
    F = np.zeros(n_dof)

    if force_bc is not None:
        for index, value in force_bc.items():
            F[index] = value

    # -------------------------
    # 2. 定义已知位移向量 d
    # -------------------------
    d = np.zeros(n_dof)

    if disp_bc is None:
        disp_bc = {}

    fixed_dofs = np.array(sorted(disp_bc.keys()), dtype=int)

    for index, value in disp_bc.items():
        d[index] = value

    # -------------------------
    # 3. 找自由自由度
    # -------------------------
    all_dofs = np.arange(n_dof)
    free_dofs = np.setdiff1d(all_dofs, fixed_dofs)

    # -------------------------
    # 4. 分块矩阵
    # -------------------------
    K_ff = K[np.ix_(free_dofs, free_dofs)]
    K_fc = K[np.ix_(free_dofs, fixed_dofs)]

    F_f = F[free_dofs]
    d_c = d[fixed_dofs]

    # -------------------------
    # 5. 求解自由位移
    # -------------------------
    rhs = F_f - K_fc @ d_c
    d_f = np.linalg.solve(K_ff, rhs)

    d[free_dofs] = d_f

    # -------------------------
    # 6. 计算支座反力
    # -------------------------
    internal_force = K @ d
    reaction = internal_force - F

    return d, reaction, free_dofs, fixed_dofs